In [ ]:
!pip install hypercorn

In [1]:
import asyncio
import hypercorn.asyncio
import hypercorn.config
from shiny import App, Inputs, Outputs, Session, render, ui



In [2]:
# 1. Gestor para iniciar/detener la app sin bloquear Jupyter
app_tasks = []

class AppTask:
    def __init__(self, app: App, port: int = 8000):
        self.shutdown_event = asyncio.Event()
        self.app = app
        self.port = port
        self.task = asyncio.create_task(self.start())
        app_tasks.append(self)

    async def start(self):
        config = hypercorn.config.Config()
        config.bind = [f"localhost:{self.port}"]
        await hypercorn.asyncio.serve(
            self.app, config, shutdown_trigger=self.shutdown_event.wait
        )

    def stop(self):
        self.shutdown_event.set()

def stop_all_apps():
    for task in app_tasks:
        task.stop()
    app_tasks.clear()



In [3]:
# 2. Define aquí tu aplicación Shiny (o impórtala desde tu archivo)
app_ui = ui.page_fluid(
    ui.input_slider("n", "Selecciona un número", 0, 100, 20),
    ui.output_text_verbatim("resultado"),
)

def server(input: Inputs, output: Outputs, session: Session):
    @render.text
    def resultado():
        return f"El doble es: {input.n() * 2}"

mi_app = App(app_ui, server)



In [4]:
# 3. Inicia la app en el puerto 8000
AppTask(mi_app, port=8000)

[2026-11-11 18:27:29 +0100] [60275] [INFO] Running on http://127.0.0.1:8000 (CTRL + C to quit)


In [ ]:
from IPython.display import IFrame

IFrame(src="http://localhost:8000", width="100%", height="400px")
